# 04 · Gradio demo (free Colab T4)

Runs the fine-tuned model behind **the same pipeline as the FastAPI service**: Pydantic validation, faithfulness check, regex guard and deterministic fallback, all imported from this repo's `app/` package. Gradio prints a public `*.gradio.live` link you can share for 72 hours.

**Runtime:** T4 GPU. **Secrets:** `HF_TOKEN` (read scope is enough).

In [ ]:
%pip install -q unsloth
%pip install -q "gradio>=5" "pydantic-settings>=2.4" "httpx>=0.27"

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/zainabbas14/slm-pii-masker.git"
if not os.path.isdir("slm-pii-masker"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
sys.path.insert(0, os.path.abspath("slm-pii-masker"))
os.environ["PROMPT_CONFIG_PATH"] = os.path.abspath("slm-pii-masker/configs/prompt_config.json")

In [ ]:
import unsloth  # noqa: F401  (must be imported before transformers)
from unsloth import FastLanguageModel

import torch
import transformers

from app.inference_server import Settings, analyze
from app.schemas import TAG_SPLIT, TAG_PATTERN

transformers.logging.set_verbosity_error()
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"

LORA_REPO = "Zain9989/pii-masker-qwen3-4b-lora"


def get_secret(name):
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


model, tokenizer = FastLanguageModel.from_pretrained(
    LORA_REPO, max_seq_length=4096, dtype=None, load_in_4bit=True, token=get_secret("HF_TOKEN"))
FastLanguageModel.for_inference(model)
STOP_IDS = [i for i in {tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id} if i is not None]

## Wrap the model as an `app` backend
`analyze()` only needs an object with an async `generate(system, user, max_tokens)` method, so the local model plugs straight into the production pipeline.

In [ ]:
class LocalUnslothBackend:
    name = f"local:{LORA_REPO}"

    async def generate(self, system: str, user: str, max_tokens: int) -> str:
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                               return_tensors="pt", return_dict=True).to("cuda")
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, temperature=None,
                                 top_p=None, top_k=None, eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id)
        return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    async def healthy(self) -> bool:
        return True

    async def aclose(self) -> None:
        return None


BACKEND = LocalUnslothBackend()
SETTINGS = Settings(backend="rules_only")   # backend object is passed explicitly; settings only toggle the guard

## UI

In [ ]:
import gradio as gr

EXAMPLES = [
    "URGENT - our dev Priya Sharma committed the production key api_9fK2mQx7LpR4tZ8vW1yN3bC6dE0gH5jA to a public "
    "repo an hour ago. Please revoke it immediately. Reach me at +1 (415) 555-0132.",
    "Hi, I was charged twice for invoice INV-20931 on my card 4111 1111 1111 1111. Please refund the duplicate. "
    "Thanks, Omar Haddad (omar.haddad@contoso.io)",
    "We received a subpoena requesting all records for employee Grace Okafor, SSN 512-44-9087. Legal needs them "
    "within 24 hours.",
    "Our webhook calls keep timing out from server 203.0.113.42 since the upgrade. Can you check rate limits?",
    "What are your office hours during the holidays?",
]


def highlight(masked: str):
    return [(part, part[1:-1]) if TAG_PATTERN.fullmatch(part) else (part, None)
            for part in TAG_SPLIT.split(masked) if part]


async def run(text: str, use_guard: bool, force_fallback: bool):
    if not text or not text.strip():
        raise gr.Error("Paste a ticket first.")
    settings = SETTINGS.model_copy(update={"enable_regex_guard": use_guard})
    backend = BACKEND
    if force_fallback:
        from app.inference_server import NoModelBackend
        backend = NoModelBackend()
    resp = await analyze(text[:6000], backend, settings)
    r, m = resp.result, resp.meta
    summary = (f"**Department:** {r.ticket_category.replace('_', ' ')}  \n"
               f"**Urgency:** {r.urgency_level}  \n"
               f"**PII found:** {', '.join(r.detected_pii_types) or 'none'}  \n"
               f"**Path:** `{m.source}`" + (f" ({m.fallback_reason})" if m.fallback_reason else "")
               + (f"  \n**Guard caught:** {', '.join(m.guard_added_types)}" if m.guard_added_types else "")
               + f"  \n**Latency:** {m.latency_ms / 1000:.2f}s")
    return highlight(r.masked_text), summary, resp.model_dump()


with gr.Blocks(title="Enterprise PII Masker") as demo:
    gr.Markdown("# Enterprise PII Masker & Ticket Router\nFine-tuned **Qwen3-4B** (QLoRA on a free T4) behind "
                "Pydantic validation and a regex safety net.")
    with gr.Row():
        with gr.Column():
            inp = gr.Textbox(label="Raw ticket", lines=8, placeholder="Paste a support ticket or HR/legal snippet…")
            guard = gr.Checkbox(value=True, label="Regex guard on model output")
            fallback = gr.Checkbox(value=False, label="Simulate model outage (rules-only fallback)")
            btn = gr.Button("Analyze", variant="primary")
            gr.Examples(EXAMPLES, inputs=inp)
        with gr.Column():
            out_hl = gr.HighlightedText(label="Masked text", combine_adjacent=False, show_legend=True)
            out_md = gr.Markdown()
            out_json = gr.JSON(label="API response")
    btn.click(run, [inp, guard, fallback], [out_hl, out_md, out_json])

demo.queue(default_concurrency_limit=1).launch(share=True, debug=False)